# NBA Champion Predictor — Exploration

This notebook handles data collection and exploratory analysis.
All scraping and feature logic lives in `src/nba_predictor/`.

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from nba_predictor.scraping import collect_all_seasons
from nba_predictor.features.selection import select_features
from nba_predictor.config import SEASON_START, SEASON_END, TRAIN_CUTOFF

## 1. Collect Data

Scrape all seasons from Basketball Reference.  
**Warning:** This takes ~6 minutes due to rate-limit sleeps.

In [ ]:
seasons_df = collect_all_seasons(start=SEASON_START, end=SEASON_END)
print(seasons_df.shape)
seasons_df.head()

## 2. Correlation Matrix

In [ ]:
pd.set_option('display.max_columns', None)

corr_matrix = seasons_df.drop(['Team', 'season'], axis=1).corr(numeric_only=True)

f = plt.figure(figsize=(19, 15))
plt.matshow(corr_matrix, fignum=f.number)
plt.xticks(range(corr_matrix.shape[1]), corr_matrix.columns, fontsize=14, rotation=45, ha='left')
plt.yticks(range(corr_matrix.shape[1]), corr_matrix.columns, fontsize=14)
cb = plt.colorbar()
cb.ax.tick_params(labelsize=14)
plt.title('Correlation Matrix', fontsize=16);

In [ ]:
corr_matrix.style.background_gradient(cmap='coolwarm').format(precision=2)

## 3. Feature Selection (train-only)

In [ ]:
selected_features = select_features(seasons_df, train_cutoff=TRAIN_CUTOFF)
print(selected_features)

all_seasons_df = seasons_df.loc[:, selected_features]
all_seasons_df.shape

## 4. Save for Modeling

Save the processed DataFrame so the modeling notebook can load it directly
without re-running the scraper.

In [ ]:
all_seasons_df.to_csv('../output/all_seasons.csv', index=False)
print('Saved to output/all_seasons.csv')